In [0]:
from pyspark.sql.functions import col, round, count, avg, sum, hour, to_timestamp

# load bronze table
trips = spark.table("nyc_taxi.bronze.nyc_taxi_trips")

# cast numeric columns
trips_clean = (
    trips
    .withColumn("fare_amount", col("fare_amount").cast("double"))
    .withColumn("tip_amount", col("tip_amount").cast("double"))
    .withColumn("total_amount", col("total_amount").cast("double"))
    .withColumn("trip_distance", col("trip_distance").cast("double"))
    .withColumn("passenger_count", col("passenger_count").cast("integer"))
    .withColumn("pickup_datetime", to_timestamp(col("pickup_datetime")))
    .filter(col("fare_amount") > 0)
    .filter(col("trip_distance") > 0)
)

print(f"Total trips: {trips_clean.count()}")

In [0]:
# trips and revenue by vendor
vendor_stats = (
    trips_clean
    .groupBy("vendor_id")
    .agg(
        count("*").alias("total_trips"),
        round(avg("fare_amount"), 2).alias("avg_fare"),
        round(avg("trip_distance"), 2).alias("avg_distance"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("tip_amount"), 2).alias("avg_tip")
    )
    .orderBy("total_trips", ascending=False)
)

vendor_stats.display()

In [0]:
# trips by hour of day
hourly_trips = (
    trips_clean
    .withColumn("hour_of_day", hour(col("pickup_datetime")))
    .groupBy("hour_of_day")
    .agg(
        count("*").alias("total_trips"),
        round(avg("fare_amount"), 2).alias("avg_fare")
    )
    .orderBy("hour_of_day")
)

hourly_trips.display()

Databricks visualization. Run in Databricks to view.

In [0]:
spark.sql("SELECT count(*) FROM nyc_taxi.bronze.nyc_taxi_trips").display()